In [3]:
import requests

from pythontraining.fachübergeifend.test import aerzte

# Einfachster GET-Aufruf -> öffentliche Test-API:
antwort = requests.get('http://httpbin.org/get')

print(f'Status Code: {antwort.status_code}')
print(f'Content-Type: {antwort.headers["Content-Type"]}')
print(f'Antwort-Größe: {len(antwort.text)} Zeichen')
print()

#JSON
daten = antwort.json()
print('Meine IP-Adresse laut API:', daten.get('origin'))
print('User-Agent:', daten.get('headers', {}).get('User-Agent'))
antwort = requests.get('http://httpbin.org/get', params={'user': 'Klinik', 'version': '1.0'})
print('URL mit Parametern:', antwort.url)
print('parameter in der Antwort:', antwort.json().get('args'))
# Timeout immer angeben sonst wartet das Script ewig wenn die API nicht antwortet:
try:
    antwort = requests.get('http://httpbin.org/delay/3', timeout=9)
    print('Antwort erhalten:', antwort.status_code)
except requests.Timeout:
    print('Fehler: Die Anfrage hat zu lange gedauert und wurde abgebrochen.')

Status Code: 200
Content-Type: application/json
Antwort-Größe: 313 Zeichen

Meine IP-Adresse laut API: 89.246.108.228
User-Agent: python-requests/2.33.1
URL mit Parametern: http://httpbin.org/get?user=Klinik&version=1.0
parameter in der Antwort: {'user': 'Klinik', 'version': '1.0'}
Antwort erhalten: 200


## Echte API: Wetterdaten abrufen
Wir nutzen jetzt die kostenlose Open-Meteo API, um aktuelle Wetterdaten für eine bestimmte Stadt abzurufen. Hier ein Beispiel, wie das funktioniert:

In [6]:
import requests  # Wichtig: mit 's' am Ende
from datetime import datetime

def wetter_abrufen(ort='Wuppertal', latitude=51.256214, longitude=7.150764):
    """ Diese Funktion ruft die aktuellen Wetterdaten von der Open-Meteo API ab. """
    url = 'https://api.open-meteo.com/v1/forecast'

    params = {
        'latitude': latitude,
        'longitude': longitude,
        'current': 'temperature_2m,wind_speed_10m,weather_code,apparent_temperature',
        'timezone': 'Europe/Berlin'
    }


    antwort = requests.get(url, params=params, timeout=10)
    antwort.raise_for_status()
    daten = antwort.json()
    aktuell = daten['current']

    return {
        'ort': ort,
        'zeitpunkt': aktuell['time'],
        'temperatur': aktuell['temperature_2m'],
        'gefuehlt': aktuell['apparent_temperature'],
        'wind': aktuell['wind_speed_10m']
    }

# Testlauf
wetter = wetter_abrufen()
for key, value in wetter.items():
    print(f'{key:<20}: {value}')

ort                 : Wuppertal
zeitpunkt           : 2026-05-07T09:00
temperatur          : 9.6
gefuehlt            : 8.0
wind                : 6.5


In [18]:
# Mehrere Städte auf einmal abrufen:
staedte = [
    ('Berlin', 52.52, 13.41),
    ('Wuppertal', 51.25, 7.15),
    ('München', 48.14, 11.58),
    ('Düsseldorf', 51.13, 6.46),
]
# Tabellen-Header drucken
print(f'{"Stadt":15} {"Temperatur":>12}')
print("-" * 28)

for name, lat, lon in staedte:
    try:
        wetter = wetter_abrufen(name, lat, lon)
        print(f'{wetter["ort"]:12} {wetter["temperatur"]:13.1f}°C')
    except requests.RequestException as e:
        print(f'Fehler beim Abrufen der Daten für {name}: {e}')

Stadt             Temperatur
----------------------------
Berlin                 8.1°C
Wuppertal              9.7°C
München               11.3°C
Düsseldorf             9.8°C


## API Daten in PostgreSQL speichern
Wetterdaten sind nur nützlich, wenn man sie über zeit vergleichen (Stichwort: grippewelle) kann. Deshalb speichern wir die Daten in einer PostgreSQL-Datenbank. Hier ein Beispiel, wie das mit der `psycopg2`-Bibliothek funktioniert:

In [24]:
import psycopg2
import requests
from datetime import datetime

Verbindung = {
    'host': 'localhost',
    'port': 5432,
    'dbname': 'klinik',
    'user': 'postgres',
    'password': 'Awb2tz',
}

# 1. Wetter-Abruf (WICHTIG: Hier nur die nackten Zahlen zurückgeben!)
def wetter_abrufen(ort, latitude, longitude):
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'current': 'temperature_2m,wind_speed_10m,weather_code,apparent_temperature',
        'timezone': 'Europe/Berlin'
    }
    antwort = requests.get(url, params=params, timeout=10)
    antwort.raise_for_status()
    daten = antwort.json()
    aktuell = daten['current']

    return {
        'ort': ort,
        'zeitpunkt': datetime.fromisoformat(aktuell['time']).strftime('%d.%m.%Y %H:%M'),
        'temperatur': aktuell['temperature_2m'], # KEIN "°C" hier!
    }

# 2. Speichern-Funktion
def wetter_speichern(wetter_daten, konfig):
    tabelle_sql = """
    CREATE TABLE IF NOT EXISTS personal.wetterdaten (
        id SERIAL PRIMARY KEY,
        abgerufen_am TIMESTAMP WITH TIME ZONE DEFAULT NOW(),
        ort TEXT NOT NULL,
        zeitpunkt TEXT,
        temperatur NUMERIC(5,2)
    );
    """
    insert_sql = """
    INSERT INTO personal.wetterdaten (ort, zeitpunkt, temperatur)
    VALUES (%(ort)s, %(zeitpunkt)s, %(temperatur)s)
    RETURNING id;
    """

    # Verbindung aufbauen
    conn = psycopg2.connect(**konfig)
    try:
        with conn: # Dieser Block sorgt für das COMMIT am Ende
            with conn.cursor() as cursor:
                cursor.execute(tabelle_sql)
                cursor.execute(insert_sql, wetter_daten)
                neue_id = cursor.fetchone()[0]
                return neue_id
    finally:
        conn.close()

# 3. Ausführung
staedte = [
    ('Berlin', 52.52, 13.41),
    ('Wuppertal', 51.25, 7.15)
]

for name, lat, lon in staedte:
    try:
        wetter = wetter_abrufen(name, lat, lon)
        eintrags_id = wetter_speichern(wetter, Verbindung)
        print(f'Gespeichert: {name} (ID: {eintrags_id}, Temp: {wetter["temperatur"]}°C)')
    except Exception as e:
        print(f'Fehler bei {name}: {e}')

Gespeichert: Berlin (ID: 3, Temp: 8.2°C)
Gespeichert: Wuppertal (ID: 4, Temp: 10.7°C)


## feitage-API - mit Authentifizierung
Viele APis verlangen einen API-Key. wir nutzen die kostenlose feiertage-API von Nager.Date, um die Feiertage für Deutschland abzurufen. Hier ein Beispiel, wie das funktioniert:

In [30]:
def feiertage_abrufen(land = 'DE', jahr = 2026):
    """ Diese Funktion ruft die Feiertage für ein bestimmtes Land und Jahr ab. """
    url = f'https://date.nager.at/api/v3/PublicHolidays/{jahr}/{land}'
    antwort = requests.get(url, timeout=10)
    antwort.raise_for_status()

    feiertage = antwort.json()
    return [
        {
            'datum': f['date'],
            'name': f['localName'],
            'national': f['global']
        }
        for f in feiertage
    ]

feiertage = feiertage_abrufen()
print(f'Feiertage in Deutschland 2026: {len(feiertage)}')
print()
for f in feiertage:
    print(f"{f['datum']}: {f['name']} National: {'Ja' if f['national'] else 'Nein Regional'}")

Feiertage in Deutschland 2026: 19

2026-01-01: Neujahr National: Ja
2026-01-06: Heilige Drei Könige National: Nein Regional
2026-03-08: Internationaler Frauentag National: Nein Regional
2026-04-03: Karfreitag National: Ja
2026-04-05: Ostersonntag National: Nein Regional
2026-04-06: Ostermontag National: Ja
2026-05-01: Tag der Arbeit National: Ja
2026-05-14: Christi Himmelfahrt National: Ja
2026-05-24: Pfingstsonntag National: Nein Regional
2026-05-25: Pfingstmontag National: Ja
2026-06-04: Fronleichnam National: Nein Regional
2026-08-15: Mariä Himmelfahrt National: Nein Regional
2026-09-20: Weltkindertag National: Nein Regional
2026-10-03: Tag der Deutschen Einheit National: Ja
2026-10-31: Reformationstag National: Nein Regional
2026-11-01: Allerheiligen National: Nein Regional
2026-11-18: Buß- und Bettag National: Nein Regional
2026-12-25: Erster Weihnachtstag National: Ja
2026-12-26: Zweiter Weihnachtstag National: Ja


In [32]:
# Feiertage in PosteSQL speichern:
Feiertage_SQL = """
    CREATE TABLE IF NOT EXISTS personal.feiertage (
    id SERIAL PRIMARY KEY,
    datum DATE NOT NULL,
    name TEXT NOT NULL,
    national BOOLEAN DEFAULT TRUE,
    land char(4) default 'DE',
        UNIQUE(datum, land)
    );
    """

Insert_SQL =\
    """
    INSERT INTO personal.feiertage (datum, name, national, land)
    VALUES (%(datum)s, %(name)s, %(national)s, 'DE')
    ON CONFLICT (datum, land) DO UPDATE
        SET name = EXCLUDED.name;
    """

with psycopg2.connect(**Verbindung) as conn:
    with conn.cursor() as cur:
        cur.execute(Feiertage_SQL)
        for f in feiertage:
            cur.execute(Insert_SQL, f)

print(f' {len(feiertage)} feiertage in der Datenbank gespeichert')
print()

print('Nächste feiertage, die relevant sind für die Schichtplanung:')
with psycopg2.connect(**Verbindung) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT datum, name FROM personal.feiertage
            WHERE datum >= CURRENT_DATE
            ORDER BY datum ASC
            LIMIT 5;
        """)
        for datum, name in cur.fetchall():
            print(f'{datum}: {name}')

 19 feiertage in der Datenbank gespeichert

Nächste feiertage, die relevant sind für die Schichtplanung:
2026-05-14: Christi Himmelfahrt
2026-05-24: Pfingstsonntag
2026-05-25: Pfingstmontag
2026-06-04: Fronleichnam
2026-08-15: Mariä Himmelfahrt


## API-Keys und headers sicher verwenden
Die miesten produktiven  APIs verlangen eine Authentifizierung über API-Keys oder Tokens. Es ist wichtig, diese sensiblen Informationen sicher zu speichern und nicht im Code zu hinterlegen.

In [36]:
import os
from dotenv import load_dotenv

load_dotenv()

# beispiel ist folgendes: APKI-Key aus .env Datei lesen

API_KEY = os.getenv('Wetter_API_KEY', 'KEIN_KEY_GESETZT')

# Headers für authentifizierte Anfragen:
headers_beispiel = {
    'Authorization': f'Bearer {API_KEY}',
    'X-API-KEY': API_KEY,
    'Accept': 'application/json',
    'User-Agent': 'KlinikSystem/1.0'
}

# test mit httpbin.org, um die Headers zu sehen:
antwort = requests.get('http://httpbin.org/headers', headers=headers_beispiel)
print('Gesendete Headers:', antwort.json().get('headers', {}))
print("*" * 50)
print("nur items")
for key, value in antwort.json().get('headers', {}).items():
    print(f'{key}: {value}')

Gesendete Headers: {'Accept': 'application/json', 'Accept-Encoding': 'gzip, deflate, zstd', 'Authorization': 'Bearer KEIN_KEY_GESETZT', 'Host': 'httpbin.org', 'User-Agent': 'KlinikSystem/1.0', 'X-Amzn-Trace-Id': 'Root=1-69fc557e-50b7004045db39083073df01', 'X-Api-Key': 'KEIN_KEY_GESETZT'}
**************************************************
nur items
Accept: application/json
Accept-Encoding: gzip, deflate, zstd
Authorization: Bearer KEIN_KEY_GESETZT
Host: httpbin.org
User-Agent: KlinikSystem/1.0
X-Amzn-Trace-Id: Root=1-69fc557e-50b7004045db39083073df01
X-Api-Key: KEIN_KEY_GESETZT


# POST-Anfrage
# Simulieren wir eine API, die einen neuen Eintrag erstellt

In [39]:
antwort = requests.post(
'http://httpbin.org/post',
json={'patient_id': 42, 'diagnose': 'j18.1', 'arzt': 'dr_weber'}, headers=headers_beispiel)
print('Status Code:', antwort.status_code)
print('Gesendete Daten:', antwort.json().get('json'))

Status Code: 200
Gesendete Daten: {'arzt': 'dr_weber', 'diagnose': 'j18.1', 'patient_id': 42}


## Robuste Fehlerbahndlung bei API-Aufrufen
APIs können aus verschiedenen Gründen fehlschlagen: Netzwerkprobleme, Serverfehler, ungültige Anfragen, etc. **Es ist wichtig**, diese Fehler **robust zu behandeln**, um Abstürze zu vermeiden und sinnvolle Fehlermeldungen zu liefern.

In [44]:
import time
import logging
import requests

logger = logging.getLogger('API-Fehlerbehandlung')

def api_aufruf_mit_retry(url, params=None, headers=None, max_try=3, wartezeit_in_sec=5):
    """
    Diese Funktion versucht, eine API-Anfrage mit einem Retry-Mechanismus durchzuführen.
    """
    for versuch in range(1, max_try + 1):
        try:
            antwort = requests.get(url, params=params, headers=headers, timeout=10)

            # 1. Spezifische Behandlung für 429 (Rate-Limiting)
            if antwort.status_code == 429:
                # Tippfehler korrigiert: 'Retry-After'
                wartezeit = int(antwort.headers.get('Retry-After', wartezeit_in_sec))
                logger.warning(f'Rate Limit erreicht. Warte {wartezeit} Sekunden... (Versuch {versuch}/{max_try})')
                time.sleep(wartezeit)
                continue # Springt sofort in den nächsten Versuch der for-Schleife

            # 2. Alle anderen HTTP-Fehler (4xx, 5xx) werfen
            antwort.raise_for_status()

            # 3. Wenn kein Fehler geworfen wurde -> Erfolg!
            return antwort

        # Fängt Verbindungsfehler, Timeouts und die HTTPError von raise_for_status() ab
        except requests.exceptions.RequestException as e:
            logger.error(f'Fehler bei API-Aufruf (Versuch {versuch}/{max_try}): {e}')

            if versuch < max_try:
                logger.info(f'Warte {wartezeit_in_sec} Sekunden bevor erneut versucht wird...')
                time.sleep(wartezeit_in_sec)
            else:
                logger.critical('Maximale Anzahl von Versuchen erreicht. API-Aufruf fehlgeschlagen.')
                raise # Wirft den Fehler endgültig, da keine Versuche mehr übrig sind



# Test
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

try:
    antwort = api_aufruf_mit_retry('http://httpbin.org/dwdwda') # Simuliert einen 429 Fehler
    print('API-Antwort erhalten:', antwort.status_code)
except Exception as e:
    print('Endgültiger Fehler:', e)

2026-05-07 12:04:38,702 - API-Fehlerbehandlung - ERROR - Fehler bei API-Aufruf (Versuch 1/3): 404 Client Error: NOT FOUND for url: http://httpbin.org/dwdwda
2026-05-07 12:04:38,705 - API-Fehlerbehandlung - INFO - Warte 5 Sekunden bevor erneut versucht wird...
2026-05-07 12:04:43,929 - API-Fehlerbehandlung - ERROR - Fehler bei API-Aufruf (Versuch 2/3): 404 Client Error: NOT FOUND for url: http://httpbin.org/dwdwda
2026-05-07 12:04:43,930 - API-Fehlerbehandlung - INFO - Warte 5 Sekunden bevor erneut versucht wird...
2026-05-07 12:04:49,179 - API-Fehlerbehandlung - ERROR - Fehler bei API-Aufruf (Versuch 3/3): 404 Client Error: NOT FOUND for url: http://httpbin.org/dwdwda
2026-05-07 12:04:49,182 - API-Fehlerbehandlung - CRITICAL - Maximale Anzahl von Versuchen erreicht. API-Aufruf fehlgeschlagen.


Endgültiger Fehler: 404 Client Error: NOT FOUND for url: http://httpbin.org/dwdwda


## Eigene mini-API mit fast-API bauen
Nicht nur konsumieren, sondern auch bereitstellen: Mit dem `fastapi`-Framework können wir schnell und einfach eine eigene REST-API erstellen, um z.B. Wetterdaten oder Schichtpläne bereitzustellen. Hier ein einfaches Beispiel:

In [45]:
# Eine einfache API für die Klinik-Daten:
# Das hier speichern wir normalerweise direkt als .py Datei und starten es mit "uvicorn dateiname:app --reload"

Der Befehl "pip" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.


In [49]:
try:
    r = requests.get("http://localhost:8000/", timeout=10)
    print("Startseite", r.json())

    r = requests.get("http://localhost:8000/aerzte", timeout=10)
    aerzte = r.json()
    print(len(aerzte), "Über api geladen")
    for a in aerzte[:3]: # nur die ersten 3 Ärzte anzeigen
        print(f"{a['id']}: {a['name']}")
except requests.exceptions.RequestException as e:
    print("Fehler beim Abrufen der API:", e)

Startseite {'status': 'API läuft', 'api': 'Klinik-API v1.0'}
8 Über api geladen
1: Dr. Anna Weber
2: Dr. Felix Klein
3: Dr. Laura Müller
